# SIMT 课后实践：通过 printf 打印线程信息

## 概述

前面已经学习了 SIMT 的线程层次、核函数写法和内置变量。本节通过一个最小实践题，让每个线程使用 `printf` 打印自己的线程信息，直观看到：**同一个 SIMT 核函数会被多个线程并发执行，而每个线程通过内置变量获得不同身份**。

### 前置要求

- 已学习 SIMT 线程架构，理解 Grid、Thread Block、Thread 的关系。
- 已学习 SIMT 核函数，理解 `<<<blocks_per_grid, threads_per_block, dyn_ubuf_size, stream>>>` 的执行配置。
- 当前环境已安装 CANN Toolkit，并具备 Ascend 950PR / Ascend 950DT NPU 运行环境。

### 学习目标

完成本练习后，你应该能够：

- 在 SIMT 核函数中使用 `printf` 打印 Device 侧信息。
- 读懂 `gridDim`、`blockDim`、`blockIdx`、`threadIdx` 这 4 个内置变量。
- 使用 `global_id = blockIdx.x * blockDim.x + threadIdx.x` 计算一维全局线程 ID。
- 观察输出顺序不固定这一现象，理解 SIMT 线程是并发执行的。

### 小节内容

- 环境准备
- 练习说明
- 编写 Device 端打印逻辑
- 编译运行
- 参考答案


## 1. 环境准备

执行下方脚本，检查 CANN Toolkit 是否可用，并把 CANN 环境变量加载到当前 notebook 进程。


In [ ]:
import os
import subprocess
import shlex
from pathlib import Path


def find_cann_home():
    candidates = []
    for key in ["ASCEND_HOME_PATH", "ASCEND_TOOLKIT_HOME"]:
        value = os.environ.get(key)
        if value:
            candidates.append(Path(value).expanduser())

    candidates.extend([
        Path.home() / "Ascend/cann",
        Path.home() / "Ascend/ascend-toolkit/latest",
        Path("/usr/local/Ascend/cann"),
        Path("/usr/local/Ascend/ascend-toolkit/latest"),
    ])

    for candidate in candidates:
        normalized = candidate
        if normalized.name in {"x86_64-linux", "aarch64-linux"}:
            normalized = normalized.parent
        set_env = normalized / "set_env.sh"
        if set_env.exists():
            return normalized.resolve(), set_env.resolve()

    raise RuntimeError("未找到 CANN Toolkit，请确认已安装 CANN，并设置环境变量。")


def source_cann_env(set_env):
    command = f"set -a && source {shlex.quote(str(set_env))} >/dev/null 2>&1 && env"
    result = subprocess.run(["bash", "-lc", command], check=True, text=True, capture_output=True)
    for line in result.stdout.splitlines():
        if "=" in line:
            key, value = line.split("=", 1)
            os.environ[key] = value


cann_home, cann_set_env = find_cann_home()
source_cann_env(cann_set_env)

print(f"CANN Toolkit: {cann_home}")


## 2. 练习说明

本练习启动一个很小的 SIMT Grid：

| 项 | 取值 |
| --- | --- |
| 线程块数 | 2 |
| 每个线程块的线程数 | 32 |
| 总线程数 | 64 |
| 核函数名 | `print_thread_info_kernel` |

要求你补全 Device 端核函数，让每个线程打印：

- `gridDim`
- `blockDim`
- `blockIdx`
- `threadIdx`
- 一维全局线程 ID `global_id`

一维全局线程 ID 的计算公式为：

```text
global_id = blockIdx.x * blockDim.x + threadIdx.x
```

由于多个线程并发执行，`printf` 输出行的先后顺序不一定按 `global_id` 从小到大排列。观察这个现象，有助于理解 SIMT 线程并发执行模型。


In [ ]:
!mkdir -p src/03_04_07_simt_practice


## 3. 编写 Device 端打印逻辑

Host 侧代码已经写好。请只补充 `print_thread_info_kernel` 中的 TODO，打印每个线程的 4 个内置变量和 `global_id`。


In [ ]:
%%writefile src/03_04_07_simt_practice/printf_thread_info.asc
// ===== 头文件引入 =====
#include <iostream>
#include "acl/acl.h"
#include "debug/asc_printf.h"

#define CHECK_ACL(x)                                                                        \
    do {                                                                                    \
        aclError __ret = x;                                                                 \
        if (__ret != ACL_ERROR_NONE) {                                                      \
            std::cerr << __FILE__ << ":" << __LINE__ << " aclError:" << __ret << std::endl; \
        }                                                                                   \
    } while (0);

// ===== Device 端核函数 =====
// TODO: 在每个线程中打印 gridDim、blockDim、blockIdx、threadIdx 和 global_id。
__global__ void print_thread_info_kernel()
{
    // TODO 1: 计算一维全局线程 ID。
    // uint32_t global_id = ...;

    // TODO 2: 使用 printf 打印 4 个内置变量和 global_id。
    // 提示：dim3 类型变量包含 x、y、z 三个分量，例如 blockIdx.x。
}

// ===== Host 端 main：已写好，无需修改 =====
int32_t main(int32_t argc, char* argv[])
{
    int32_t device_id = 0;
    aclrtStream stream = nullptr;

    // 初始化 ACL，选择当前进程要使用的 Device，并创建 stream。
    CHECK_ACL(aclInit(nullptr));
    CHECK_ACL(aclrtSetDevice(device_id));
    CHECK_ACL(aclrtCreateStream(&stream));

    // 核函数启动配置：启动 2 个线程块，每个线程块 32 个线程。
    // 因此本练习一共会启动 2 * 32 = 64 个线程。
    uint32_t blocks_per_grid = 2;
    uint32_t threads_per_block = 32;
    uint32_t dyn_ubuf_size = 0;  // 本练习不使用动态共享内存。

    // 使用 <<<blocks_per_grid, threads_per_block, dyn_ubuf_size, stream>>> 启动 SIMT 核函数。
    // 这 4 个执行配置参数会决定核函数内 gridDim、blockDim、blockIdx、threadIdx 的取值。
    print_thread_info_kernel<<<blocks_per_grid, threads_per_block, dyn_ubuf_size, stream>>>();

    // 核函数异步下发到 stream 后，需要同步等待 Device 侧 printf 完成。
    CHECK_ACL(aclrtSynchronizeStream(stream));

    // 释放 stream，复位 Device，并完成 ACL 去初始化。
    CHECK_ACL(aclrtDestroyStream(stream));
    CHECK_ACL(aclrtResetDevice(device_id));
    CHECK_ACL(aclFinalize());
    return 0;
}


创建 CMake 配置。


In [ ]:
%%writefile src/03_04_07_simt_practice/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)

find_package(ASC REQUIRED)

project(simt_printf_practice LANGUAGES ASC CXX)

add_executable(demo
    printf_thread_info.asc
)

set(CMAKE_ASC_ARCHITECTURES "dav-3510" CACHE STRING "NPU ARCH, e.g. dav-3510")
target_compile_options(demo PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=${CMAKE_ASC_ARCHITECTURES} --enable-simt>
)


## 4. 编译运行

补全 Device 端代码后，在 NPU 环境中执行以下命令编译并运行。


In [ ]:
# 需在已配置 CANN 环境的 NPU 机器上执行
!cd src/03_04_07_simt_practice && mkdir -p build && cd build && \
 cmake -DCMAKE_ASC_ARCHITECTURES=dav-3510 .. && make -j && \
 ./demo


运行成功后会看到 64 行线程信息。输出顺序可能每次运行都不完全一致，但每一行都会包含类似下面的信息：

```text
gridDim=(2,1,1), blockDim=(32,1,1), blockIdx=(0,0,0), threadIdx=(0,0,0), global_id=0
```

你可以重点观察：

- 所有线程看到的 `gridDim` 和 `blockDim` 相同，因为它们来自 Host 侧执行配置。
- 不同线程的 `blockIdx` 和 `threadIdx` 不同，因为它们表示当前线程所属线程块和块内位置。
- `global_id` 把 `blockIdx.x` 和 `threadIdx.x` 组合成一维连续编号。
- 打印顺序不保证按 `global_id` 排列，体现了线程并发执行。


## 5. 参考答案

可以执行下面的单元格查看参考实现。建议先独立完成，再对照检查 `global_id` 计算和 `printf` 参数顺序。


In [ ]:
!cat answer/03.04.07_printf_thread_info.asc


In [ ]:
!cat answer/03.04.07_CMakeLists.txt


## 小结

本练习通过 `printf` 直接观察每个线程的身份信息，串起了 SIMT 编程模型中的几个核心点：

- Host 侧通过 `<<<blocks_per_grid, threads_per_block, dyn_ubuf_size, stream>>>` 决定 Grid 和 Thread Block 的规模。
- Device 侧每个线程都执行同一份核函数代码。
- 每个线程通过 `gridDim`、`blockDim`、`blockIdx`、`threadIdx` 获取自己的执行上下文。
- 一维全局线程 ID 可以用 `blockIdx.x * blockDim.x + threadIdx.x` 计算得到。
- 多线程并发执行时，`printf` 的输出顺序不代表线程编号顺序。
